# 03 — Exploratory Data Analysis
Distributions, correlations, outlier boxplots, age-group breakdowns.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', font_scale=1.1)
print('Ready.')

Ready.


## 1. Load clean data

In [2]:
df = pd.read_csv('../../data/processed/bioage_final_clean.csv')
print(f'Clean data: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

Clean data: 22094 rows x 13 columns


,Age,LBXSAL,LBXSCR,LBXGLU,CRP,LBXLYPCT,LBXMCVSI,LBXRDW,LBXSAPSI,LBXWBCSI,LBXGH,LBDHDD,LBXTC
0,44.0,3.5,0.8,90.0,2.44,35.8,80.1,13.7,74.0,5.3,6.0,39.0,105.0
1,70.0,5.0,1.2,157.0,0.05,29.4,90.3,12.5,48.0,7.5,7.1,59.0,147.0
2,16.0,4.2,0.9,84.0,0.09,30.0,99.0,14.5,41.0,6.6,4.7,54.0,147.0
3,73.0,3.9,1.2,100.0,0.21,29.1,88.6,13.4,77.0,6.6,5.9,49.0,186.0
4,16.0,4.5,0.7,91.0,0.01,31.3,87.6,12.7,104.0,4.8,5.0,53.0,126.0


## 2. Feature distributions

In [3]:
feature_cols = [c for c in df.columns if c != 'Age']
n_features = len(feature_cols)
n_cols_plot = 3
n_rows_plot = (n_features + n_cols_plot - 1) // n_cols_plot

fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(14, n_rows_plot * 3.5))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    data = df[col].dropna()
    ax.hist(data, bins=40, edgecolor='white', alpha=0.8, color='steelblue')
    ax.set_title(col, fontsize=11)
    skew = data.skew()
    ax.text(0.95, 0.95, f'skew={skew:.1f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions (Clean Data)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../../reports/figures/06_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/06_distributions.png')

Saved: reports/figures/06_distributions.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_19528\505560087.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Correlation heatmap

In [4]:
corr = df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('Correlation Matrix — All Features + Age')
plt.tight_layout()
plt.savefig('../../reports/figures/07_correlation_heatmap.png', dpi=150)
plt.show()
print('Saved: reports/figures/07_correlation_heatmap.png')

C:\Users\vinit\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\matrix.py:260: FutureWarning: Format strings passed to MaskedConstant are ignored, but in future may error or produce different behavior
  annotation = ("{:" + self.fmt + "}").format(val)


Saved: reports/figures/07_correlation_heatmap.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_19528\1911911005.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Top correlations with Age
age_corr = corr['Age'].drop('Age').sort_values(key=abs, ascending=False)
print('Correlations with Age (sorted by |r|):')
print(age_corr.to_string())

Correlations with Age (sorted by |r|):
LBXGH       0.333968
LBXSAPSI   -0.280857
LBXSCR      0.275593
LBXGLU      0.268742
LBXTC       0.261690
LBXMCVSI    0.249798
LBXSAL     -0.222727
LBXLYPCT   -0.214701
LBXRDW      0.162824
CRP         0.097639
LBDHDD      0.063501
LBXWBCSI   -0.000240


## 4. Boxplots per biomarker

In [6]:
fig, axes = plt.subplots(1, len(feature_cols), figsize=(4 * len(feature_cols), 5))
if len(feature_cols) == 1:
    axes = [axes]

for i, col in enumerate(feature_cols):
    ax = axes[i]
    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='navy'),
               medianprops=dict(color='red'))
    ax.set_title(col, fontsize=9, rotation=45)
    ax.set_xticklabels([])

fig.suptitle('Boxplots — Candidate Features', fontsize=14)
plt.tight_layout()
plt.savefig('../../reports/figures/08_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/08_boxplots.png')

Saved: reports/figures/08_boxplots.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_19528\1459248796.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Age-group breakdowns

In [7]:
bins = [0, 18, 30, 45, 60, 75, 100]
labels = ['0-17', '18-29', '30-44', '45-59', '60-74', '75+']
df['age_group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

# Top 4 features most correlated with Age
top4 = age_corr.head(4).index.tolist()
print(f'Top 4 Age-correlated features: {top4}')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(top4):
    ax = axes[i // 2][i % 2]
    order = labels
    sns.boxplot(data=df, x='age_group', y=col, order=order, ax=ax,
                palette='viridis', fliersize=2)
    ax.set_title(f'{col} by Age Group')
    ax.set_xlabel('Age Group')

fig.suptitle('Top Biomarkers Stratified by Age Group', fontsize=14)
plt.tight_layout()
plt.savefig('../../reports/figures/09_age_group_breakdowns.png', dpi=150)
plt.show()
print('Saved: reports/figures/09_age_group_breakdowns.png')

C:\Users\vinit\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
C:\Users\vinit\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
C:\Users\vinit\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain

Top 4 Age-correlated features: ['LBXGH', 'LBXSAPSI', 'LBXSCR', 'LBXGLU']
Saved: reports/figures/09_age_group_breakdowns.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_19528\613287069.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Feature vs Age scatter plots

In [8]:
n_feat = len(feature_cols)
n_cols = 3
n_rows = (n_feat + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.array(axes).flatten() if n_feat > 1 else [axes]

for i, col in enumerate(feature_cols):
    ax = axes[i]
    ax.scatter(df['Age'], df[col], alpha=0.15, s=5, color='steelblue')
    # Add regression line
    mask = df[col].notna() & df['Age'].notna()
    if mask.sum() > 10:
        slope, intercept, r, p, se = stats.linregress(df.loc[mask, 'Age'], df.loc[mask, col])
        x_line = np.linspace(df['Age'].min(), df['Age'].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color='red', linewidth=2,
                label=f'r={r:.3f}')
        ax.legend(fontsize=9)
    ax.set_xlabel('Age')
    ax.set_ylabel(col)
    ax.set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Features vs Age with Linear Fit', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../../reports/figures/10_feature_vs_age.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/10_feature_vs_age.png')

Saved: reports/figures/10_feature_vs_age.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_19528\3463161576.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Summary statistics table

In [9]:
summary = df.describe().round(3).T
summary['skew'] = df[feature_cols + ['Age']].skew().round(2)
summary['kurtosis'] = df[feature_cols + ['Age']].kurtosis().round(2)
summary

,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
Age,22094.0,42.965,19.962,12.00,25.00,43.0,60.00,85.0,0.06,-1.24
LBXSAL,22094.0,4.157,0.368,1.50,3.90,4.2,4.40,5.5,-0.49,1.24
LBXSCR,22094.0,0.844,0.299,0.20,0.68,0.8,0.96,5.0,5.80,68.24
LBXGLU,22094.0,107.899,34.545,40.00,93.00,100.0,109.00,500.0,4.57,27.77
CRP,22094.0,2.432,6.272,0.01,0.20,0.7,2.32,188.5,10.16,165.66
LBXLYPCT,22094.0,31.599,8.832,2.90,25.50,31.1,37.10,80.0,0.35,0.35
LBXMCVSI,22094.0,88.270,5.923,50.80,85.30,88.7,91.90,120.0,-0.91,3.14
LBXRDW,22094.0,13.407,1.336,9.70,12.60,13.2,13.90,25.0,2.39,11.01
LBXSAPSI,22094.0,87.069,53.029,16.00,60.00,74.0,94.00,500.0,3.53,16.41
LBXWBCSI,22094.0,6.741,2.049,1.60,5.40,6.4,7.80,50.0,1.87,17.83


## EDA Summary
See saved charts in `reports/figures/` for presentation. Key findings:
1. **Age is top-coded at 80** — spike removed during cleaning.
2. **Right-skewed features**: CRP, LBXSAPSI, LBXWBCSI — log transforms may help.
3. **Strongest Age correlations**: [to be filled after running].
4. **Multicollinearity**: some CBC-derived features are highly correlated (r > 0.8).
5. **Age-group patterns**: biomarker distributions shift with age, some non-linearly.
